# Gradio로 AI 웹 데모 만들기

- Gradio는 머신러닝 모델의 웹 데모를 만들기 위해 주로 사용하는 Python 기반 오픈소스 라이브러리로, 프론트엔드 코드 없이 웹 UI를 구현할 수 있다.
- `Interface()` 클래스를 활용하여 GUI를 구현할 수 있다.
- Gradio는 다양한 인터페이스 타입을 제공하여 목적에 따라 손쉽게 웹 UI를 만들 수 있다.
- `Interface`, `ChatInterface`, `TabbedInterface`, `Blocks`는 각각 사용 목적과 설계 방식이 다르다.
    - `Interface`: 함수 하나를 입력/출력 컴포넌트에 연결하는 가장 간단한 방식
    - `ChatInterface`: 챗봇 UI를 빠르게 구성
    - `TabbedInterface`: 여러 Interface를 탭으로 묶어 하나의 앱으로 구성
    - `Blocks`: 레이아웃과 이벤트를 직접 제어하는 저수준 API

### 목차
1. 필요한 라이브러리 설치하기
2. Gradio 프로그램 실행하기
3. Gradio 인터페이스: `Interface`, `ChatInterface`, `TabbedInterface`, `Blocks`
4. 간단한 Gradio ChatBot 만들기
5. Gradio + OpenAI API ChatBot 만들기

## 장점

- **웹 개발 코드가 필요 없음**
    - HTML/CSS/JS 없이 UI 생성 가능
- **Python 함수만 알면 됨**
    - 기초 Python만 배우면 바로 웹 앱 제작 가능
- **AI 모델과 바로 연결 가능**
    - OpenAI API, Gemini, LangChain 등과 바로 연결 가능

```python
def chatbot(message):
    return llm.invoke(message)

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text"
)
```

## 1. 필요한 라이브러리 설치하기

이 자료는 **Gradio 6.5.1** 기준이다.

In [ ]:
%pip install -q "gradio==6.5.1" openai

In [1]:
import gradio as gr
print(gr.__version__)
print(gr.__file__)  # 어느 경로에서 로드되는지 확인

c:\myenv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


6.30.0
c:\myenv\.venv\Lib\site-packages\gradio\__init__.py


## 2. Gradio 프로그램 실행하기

인터페이스에서 이벤트가 발생하면 실행되는 함수(`fn`)를 만들고, `Interface`로 웹 UI와 연결한 뒤 `launch()`로 실행한다.

In [ ]:
import gradio as gr

# 인터페이스에서 이벤트가 발생되면 실행되는 함수
def greet(name):
    return f"Hello {name}!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")

demo.launch(share=True)  # 외부 서버에서 접속하려면 server_name="0.0.0.0"

**이 코드를 실행한 결과** (내 서버 IP가 `11.22.33.44`라고 가정)

- 서버 내부: `0.0.0.0:7860` (`localhost:7860`, `127.0.0.1:7860`)
- 서버 외부: `11.22.33.44:7860`으로 접속해서 확인
- 7860 포트부터 시작해서, 이미 사용 중이면 1씩 증가하면서 실행된다.

> 한 노트북에서 여러 데모를 연달아 실행할 때는 이전 데모를 `demo.close()`로 닫아 두면 포트가 정리된다.

In [6]:
demo.close()      # 이 데모만 종료
gr.close_all()    # 실행 중인 모든 Gradio 데모 종료

Closing server running on port: 7860


**입력이 여러 개, 출력이 여러 개인 경우**

- 함수의 매개변수 순서 = `inputs` 리스트의 순서
- 함수의 리턴 순서 = `outputs` 리스트의 순서

In [ ]:
import gradio as gr

# 여기서의 input이 인터페이스의 inputs (순서대로)
def order(name, is_iced, shots):
    drink = "아이스 아메리카노" if is_iced else "따뜻한 아메리카노"
    message = f"{name}님, {drink} (샷 {shots}개) 주문이 접수되었습니다."
    price = 4000 + (shots - 1) * 500   # 기본 4,000원, 샷 추가 시 500원씩
    return message, price              # 리턴은 인터페이스의 outputs (순서대로)

demo = gr.Interface(
    fn=order,
    inputs=[
        gr.Textbox(label="이름"),
        gr.Checkbox(label="아이스로 주문"),
        gr.Slider(1, 5, step=1, value=1, label="샷 개수"),
    ],
    outputs=[
        gr.Textbox(label="주문 확인"),
        gr.Number(label="결제 금액(원)"),
    ],
)

demo.launch(share=True)

## 3. Gradio 인터페이스

| 인터페이스 종류 | 설명 | 주요 사용 사례 | 커스터마이징 수준 |
| --- | --- | --- | --- |
| **`Interface`** | 가장 기본적인 UI 인터페이스 | 이미지 분류, 텍스트 입력 → 출력 | 낮음 (빠른 개발용) |
| `ChatInterface` | 채팅형 인터페이스에 특화된 UI | 챗봇, LLM 응답형 앱 | 중간 (채팅용 구조 제공) |
| `TabbedInterface` | 여러 인터페이스를 탭으로 구성 | 다양한 기능 분할 제공 | 낮음~중간 |
| **`Blocks`** | 가장 유연한 구성 방식 (모듈형) | 복잡한 앱, 레이아웃 커스터마이징 | 매우 높음 (자유로운 레이아웃 구성) |

### 3-1. Interface

- `Interface`는 Python 함수를 웹 애플리케이션으로 자동 변환해주는 가장 기본적인 UI 구성 도구이다.
- `inputs`, `outputs` 타입만 지정하면 UI가 생성된다.
    - 웹 개발을 몰라도 **AI 웹 UI를 자동으로 만들어주는 도구**
- **"입력(Input) → 함수(Function) → 출력(Output)"** 구조를 가진 가장 단순한 AI 웹앱 제작 방식이다.
    - 사용자가 값을 입력하면 → 파이썬 함수가 실행되고 → 결과가 화면에 출력된다.
- 사용하기 매우 쉬우며, 초보자에게 적합하다.

In [ ]:
import gradio as gr

def greet(name):                  # 사용자 정의 함수, 입력값 : name
    return f"Hello, {name}!"      # 출력값 : return 뒤의 문장

demo = gr.Interface(fn=greet, inputs="text", outputs="text")  # 이 함수(fn)를 웹과 연결해줌
demo.launch(share=True)

#### 파라미터 설명

| 파라미터 | 의미 | 설명 |
| --- | --- | --- |
| `fn` | 실행할 함수 | 사용자의 입력을 받아 실행됨 |
| `inputs` | 입력 타입 | 어떤 UI 컴포넌트를 만들지 결정 |
| `outputs` | 출력 타입 | 결과를 어떤 형태로 보여줄지 결정 |

#### launch() 역할

- 로컬 서버 실행
- 웹 브라우저 자동 열기
- **`share=True`**
    - 외부 접속 가능한 링크 생성
    - 다른 사람들과 공유 가능

#### 실행 흐름

사용자 입력 → `greet()` 함수 실행 → 결과 반환 → 웹 화면 출력

1. 사용자가 텍스트 입력
2. Gradio가 입력값을 `greet(name)`에 전달
3. 함수 실행
4. return 값이 자동으로 output 박스에 표시

### 3-2. ChatInterface

- 챗봇을 위한 **특화된 인터페이스**
- 메시지 히스토리를 자동 관리
- 함수는 `fn(message, history)` 형태이며, **응답 문자열만 반환**하면 된다. (히스토리는 Gradio가 알아서 갱신)
- 내부적으로 `Chatbot` 컴포넌트를 사용하며, 별도 `Blocks`로 구성하지 않아도 자동으로 구성된다.

In [ ]:
import gradio as gr

def respond(message, history):
    return "안녕하세요!"

gr.ChatInterface(fn=respond).launch(share=True)

### 3-3. TabbedInterface

- 여러 개의 `Interface`나 `Blocks`를 탭으로 묶어주는 컴포넌트
- 각각의 기능을 탭으로 분리해 하나의 웹 앱 안에 정리해서 보여줄 수 있다.
- `Interface`는 하나의 기능 = 하나의 화면
- `TabbedInterface`는 여러 개의 기능 = 여러 화면을 탭으로 묶음

```python
gr.TabbedInterface(
    [tab1, tab2],
    ["뒤집기", "대문자"]
).launch()
```

| 부분 | 의미 |
| --- | --- |
| `[tab1, tab2]` | 묶을 Interface 목록 |
| `["뒤집기", "대문자"]` | 각 탭의 이름 |
| `.launch()` | 실행 |

In [ ]:
import gradio as gr

tab1 = gr.Interface(lambda x: x[::-1], inputs="text", outputs="text", title="Reverse")
tab2 = gr.Interface(lambda x: x.upper(), inputs="text", outputs="text", title="Uppercase")

gr.TabbedInterface([tab1, tab2], ["뒤집기", "대문자"]).launch(share=True)

### 3-4. Blocks

- Gradio에서 가장 강력하고 **유연한 인터페이스**로, **개발자가 UI 동작을 직접 설계하는 방식**
    - 레이아웃 배치 → 버튼 **이벤트** 정의 → 입출력 흐름 설계 → 여러 **이벤트** 연결
- 레이아웃 배치, 이벤트 정의, 입력/출력 흐름 제어를 직접 설계할 수 있어 복잡한 AI 애플리케이션 제작에 필수적인 구조이다.
    - 레이아웃(행/열), 사용자 정의 이벤트, 복합 입력 구조 등을 자유롭게 구성 가능
    - 여러 컴포넌트를 조합해 복잡한 앱을 만들 수 있음

#### 한 개의 위젯

In [ ]:
import gradio as gr

with gr.Blocks() as demo:     # 하나의 앱 컨테이너 생성, 이 안에 모든 UI 컴포넌트를 배치
    txt = gr.Textbox(label="입력")
    btn = gr.Button("출력")
    out = gr.Textbox(label="출력")

    def echo(x):
        return x              # 단순 입력값 반환 함수

    btn.click(echo, txt, out)  # event binding: echo 실행 -> txt 값을 입력으로 전달 -> 반환값을 out에 출력

demo.launch(share=True)

| 변수 | 컴포넌트 | 역할 |
| --- | --- | --- |
| `txt` | 입력창 | 사용자 입력 |
| `btn` | 버튼 | 이벤트 트리거 |
| `out` | 출력창 | 결과 표시 |

**이벤트 연결 (Event Binding)**

```python
# 버튼.click(실행함수, 입력컴포넌트, 출력컴포넌트)
btn.click(echo, txt, out)
```

버튼 클릭 시 → `echo` 함수 실행 → `txt` 값을 입력으로 전달 → 반환값을 `out`에 출력

**UI 배치, 입력/출력 흐름, 이벤트 체이닝 등도 제어할 수 있다.**

- 가로 배치: `gr.Row()`

In [ ]:
import gradio as gr

def on_click(x):
    return x

with gr.Blocks() as demo:
    with gr.Row():
        txt = gr.Textbox(label="입력")
        btn = gr.Button("클릭")
        out = gr.Textbox(label="출력")

    btn.click(on_click, inputs=txt, outputs=out)

demo.launch(share=True)

- 세로 배치: `gr.Column()`

In [ ]:
import gradio as gr

def on_click(x):
    return x

with gr.Blocks() as demo:
    with gr.Column():
        txt = gr.Textbox(label="입력")
        btn = gr.Button("클릭")
        out = gr.Textbox(label="출력")

    btn.click(on_click, inputs=txt, outputs=out)

demo.launch(share=True)

#### 여러 개의 위젯

입력이 여러 개면 `inputs=[...]`, 출력이 여러 개면 `outputs=[...]`로 리스트를 넘긴다. 함수의 매개변수/리턴 순서가 리스트 순서와 대응된다.

In [ ]:
import gradio as gr

with gr.Blocks() as bl:
    txt = gr.Textbox()
    chk = gr.Checkbox(label="대문자로 바꾸기")
    btn = gr.Button("클릭")
    out = gr.Textbox(label="출력1")
    out2 = gr.Textbox(label="출력2-UpperCase")

    def on_click(x, to_upper):
        upp = x.upper() if to_upper else x
        return x, upp

    btn.click(on_click, inputs=[txt, chk], outputs=[out, out2])

bl.launch(share=True)

## 4. 간단한 ChatBot 만들기

`Blocks`로 `Chatbot`, `Textbox`, `Button`을 조합해 직접 챗봇을 만든다. 응답은 미리 정해진 문장 중 하나를 골라 한 글자씩 스트리밍한다.

In [9]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
os.getenv("TAVILY_API_KEY")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

사용 모델: gpt-6-luna


In [ ]:
#ChatInterface 예제
import gradio as gr
from openai import OpenAI

client = OpenAI()
last_id = {}   # 세션별 직전 응답 ID

def respond(message, history, request: gr.Request):
    session_id = request.session_hash

    if not history:                              # 첫 대화/초기화면 이전 대화 연결을 끊음
        last_id.pop(session_id, None)

    stream = client.responses.create(
        model=default_model,
        instructions="당신은 친절한 AI 챗봇입니다.",   # 매 호출마다 다시 넘겨야 함
        input=message,                                 # 이번 메시지만 전달
        previous_response_id=last_id.get(session_id),  # 첫 호출은 None
        reasoning={"effort": "none"},
        stream=True,
    )

    full_response = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            full_response += event.delta
            yield full_response
        elif event.type == "response.completed":
            last_id[session_id] = event.response.id    # 다음 턴에 쓸 ID 저장

gr.ChatInterface(
    fn=respond,
    textbox=gr.Textbox(placeholder="메세지를 입력하세요...", label="입력"),
).launch(share=True)

In [ ]:
#BlockInterface 예제
import uuid
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

llm = ChatOpenAI(model=default_model, reasoning_effort="none")

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 챗봇입니다."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()

# 세션별 히스토리 저장소
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def respond(msg, chat_history, session_id):
    full_response = ""
    updated_history = chat_history + [{"role": "user", "content": msg}]
    config = {"configurable": {"session_id": session_id}}   # 히스토리는 이 config로 연결

    for token in chat_chain.stream({"input": msg}, config=config):
        full_response += token
        yield updated_history + [{"role": "assistant", "content": full_response}], ""

    yield updated_history + [{"role": "assistant", "content": full_response}], ""

with gr.Blocks() as bl:
    session_id = gr.State(lambda: str(uuid.uuid4()))   # 접속(세션)마다 새 ID
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="메세지를 입력하세요...", label="입력")
    clear_btn = gr.Button("초기화")

    msg.submit(respond, inputs=[msg, chatbot, session_id], outputs=[chatbot, msg])
    # 초기화: 화면을 비우고 새 세션 ID를 발급해서 LLM의 기억도 함께 초기화
    clear_btn.click(lambda: ([], "", str(uuid.uuid4())), outputs=[chatbot, msg, session_id])

bl.queue().launch(share=True)

### Block을 사용한 챗봇 설명

- **`respond`**: 메시지를 제출하면 호출되는 사용자 정의 함수. 스트리밍 출력을 위해 **`yield`**로 부분 응답을 내보낸다.
- **`gr.Chatbot()`**: 대화 히스토리를 시각적으로 표시하는 Gradio 채팅 UI 컴포넌트
    - Gradio 6.x에서는 `[{"role": "user" | "assistant", "content": "..."}, ...]` 형식(messages 형식)의 리스트를 입력으로 받아 대화 형태로 렌더링한다.
    - 텍스트 기반 챗봇, LLM 인터페이스, 고객센터 UI 등에 사용
    - `chat_history`: `gr.Chatbot()`이 사용하는 대화 내역 리스트. 이 값을 업데이트하면 챗봇 UI도 자동으로 갱신된다.
- **`msg.submit`**: 사용자가 메시지 입력 후 엔터를 누르면 실행할 함수와 입출력을 연결한다.
- **`clear_btn.click`**: 대화 초기화 버튼 기능을 담당한다.
- **`queue()`**: 요청을 순서대로 비동기 처리하도록 도와준다.
- **`launch(share=True)`**: 외부에서 접속 가능한 링크를 생성해 앱을 공유한다.

| 매개변수 | 설명 |
| --- | --- |
| `msg` | 사용자가 방금 입력한 질문 |
| `chat_history` | 지금까지의 대화 목록 (리스트) |
| `chat_history + [{...}]` | 기존 내역에 새 메시지를 추가 |

### Gradio 내부 동작 흐름 요약

1. 사용자가 메시지 입력 → `msg`와 `chat_history`가 `respond()`로 전달됨
2. `respond()` 함수에서 응답을 생성하고 `chat_history`에 추가
3. 업데이트된 `chat_history`를 반환(`yield`) → `gr.Chatbot()`이 자동으로 대화 창 갱신

**관련 팁**
- `yield ...` : 실시간(스트리밍) 응답 출력
- `history.copy()` : 대화 내역을 보존할 때는 복사본 저장 권장

#### queue() 가 필요한 경우

| 상황 | queue() 필요 여부 |
| --- | --- |
| `yield` 사용한 스트리밍 챗봇 | ✅ 필수 |
| `time.sleep` 등 지연 처리 | ✅ 필수 |
| 여러 사용자 동시 요청 제어 | ✅ 필수 |
| 단순 즉시 출력 앱 (예: 계산기) | ❌ 없어도 됨 |

| 항목 | 설명 |
| --- | --- |
| `queue()` 호출 안 함 | 모든 함수가 **즉시 동기적으로 실행됨** |
| `queue()` 호출 | 함수가 **작업 큐에 등록되어 비동기 처리 가능**, `yield`도 지원됨 |

## 5. Gradio + OpenAI API를 사용한 챗봇 만들기

앞의 챗봇에서 "랜덤 응답" 부분을 OpenAI API 스트리밍 호출로 바꾼다. 사용 모델은 `openai-6-luna`이다.

In [ ]:
import os
from getpass import getpass

# API 키는 환경변수로 관리한다. (코드/노트북에 직접 적지 않기)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

default_model = "openai-6-luna"

In [ ]:
# OpenAI API를 사용한 스트리밍 챗봇 예제
import gradio as gr
from openai import OpenAI

client = OpenAI()

def respond(msg, chat_history):
    # 1. message와 history를 OpenAI API에 맞는 포맷으로 변환
    messages = [{"role": "system", "content": "당신은 친절한 AI 챗봇입니다."}]

    for chat in chat_history:
        if isinstance(chat, dict) and "role" in chat:
            messages.append({"role": chat["role"], "content": chat["content"]})
    messages.append({"role": "user", "content": msg})

    full_response = ""
    updated_history = chat_history + [{"role": "user", "content": msg}]

    # 2. OpenAI API 호출
    # stream은 list가 아니라 generator iterator: 한 번 순회하면 내부 상태가 소비되어 다시 순회할 수 없다.
    stream = client.chat.completions.create(
        model=default_model,
        messages=messages,
        stream=True,        # 스트리밍 모드 활성화
        max_tokens=500,
        temperature=0.5     # 응답의 다양성을 조절하는 온도 파라미터
    )

    # 3. 청크가 도착할 때마다 화면 갱신
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            full_response += chunk.choices[0].delta.content
            yield updated_history + [{"role": "assistant", "content": full_response}], ""

    yield updated_history + [{"role": "assistant", "content": full_response}], ""

with gr.Blocks() as bl:
    chatbot = gr.Chatbot()  # Gradio 6.x: type 인자 불필요 (messages 형식이 기본)
    msg = gr.Textbox(placeholder="메세지를 입력하세요...", label="입력")
    clear_btn = gr.Button("초기화")

    msg.submit(respond, inputs=[msg, chatbot], outputs=[chatbot, msg])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg])

bl.queue().launch(share=True)

## 정리

| 상황 | 추천 |
| --- | --- |
| 함수 하나를 빠르게 웹으로 | `Interface` |
| LLM 챗봇을 몇 줄로 | `ChatInterface` |
| 여러 기능을 한 앱에 | `TabbedInterface` |
| 레이아웃/이벤트를 직접 설계 | `Blocks` |